### DistilBERT Fine-Tuning for Jailbreak Detection

Tier 3 (Transformer) of the two-arm experimental design:
- **Arm A** - trained on single-turn WildGuardMix data
- **Arm B** - trained on multi-turn SafeDialBench + LMSYS data

Both arms are evaluated against the **shared multi-turn test set T1**.

Primary metric: **Unsafe Recall @ Precision = 0.95**

### Section 1 - Imports & Configuration

In [ ]:
%pip install -q "accelerate>=1.1.0"
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, auc, confusion_matrix,
    f1_score, precision_recall_curve, roc_auc_score, roc_curve,
)
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments,
)

warnings.filterwarnings("ignore")

# Path anchors
PROCESSED_DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
REPORTS_DIR = Path("../reports")
FIGURES_DIR = REPORTS_DIR / "figures"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
SEEDS = [42, 1234, 2025]

# Device
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Seeds: {SEEDS}")

### Section 2 - Data Loading

Loads train/val splits per arm from CSV files produced by `scripts/train_test_split.py`.

**Prerequisite**: Run `scripts/train_test_split.py` before executing this section.

In [ ]:
def load_split(dataset: str, split: str) -> pd.DataFrame:
    """Load X and Y CSVs for a given dataset arm and split."""
    X = pd.read_csv(PROCESSED_DATA_DIR / f"{dataset}_X_{split}.csv")
    y = pd.read_csv(PROCESSED_DATA_DIR / f"{dataset}_Y_{split}.csv")
    df = pd.concat([X, y], axis=1)
    df["dataset"] = dataset
    df["split"] = split
    return df

# Arm A: Single-turn (WildGuardMix)
train_a = load_split("singleturn", "train")
val_a   = load_split("singleturn", "val")

# Arm B: Multi-turn (SafeDialBench + LMSYS)
train_b = load_split("multiturn", "train")
val_b   = load_split("multiturn", "val")

# Shared test set T1 - both arms evaluate on same multi-turn test set
test_t1 = load_split("multiturn", "test")

# Sanity checks
for name, df in [("train_a", train_a), ("val_a", val_a),
                  ("train_b", train_b), ("val_b", val_b), ("test_t1", test_t1)]:
    harm_rate = df["harm"].mean()
    print(f"{name:10s}  rows={len(df):5d}  harm_rate={harm_rate:.3f}")

### Section 3 - Tokenization & PyTorch Dataset

Uses DistilBERT tokenizer with default **right-side truncation**.
Revisit with left-truncation only if multi-turn results underperform.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class JailbreakDataset(torch.utils.data.Dataset):
    """PyTorch Dataset wrapper for tokenized conversation text."""

    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = MAX_LENGTH):
        texts = df["conversation"].astype(str).tolist()
        labels = df["harm"].astype(int).tolist()
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


train_dataset_a = JailbreakDataset(train_a, tokenizer)
val_dataset_a   = JailbreakDataset(val_a,   tokenizer)
train_dataset_b = JailbreakDataset(train_b, tokenizer)
val_dataset_b   = JailbreakDataset(val_b,   tokenizer)
test_dataset_t1 = JailbreakDataset(test_t1, tokenizer)

print(f"Arm A - train: {len(train_dataset_a):,}  val: {len(val_dataset_a):,}")
print(f"Arm B - train: {len(train_dataset_b):,}  val: {len(val_dataset_b):,}")
print(f"T1 test      - {len(test_dataset_t1):,}")

### Section 4 - Metrics & Trainer Setup

In [ ]:
def compute_metrics(eval_pred):
    """Callback for HuggingFace Trainer: accuracy, F1, ROC-AUC."""
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_unsafe": f1_score(labels, preds, pos_label=1, average="binary"),
        "roc_auc": roc_auc_score(labels, probs[:, 1]),
    }

In [ ]:
def train_arm(train_dataset, val_dataset, seed: int, arm_name: str) -> Trainer:
    """
    Fine-tune DistilBERT on the given arm.

    Parameters
    ----------
    train_dataset : JailbreakDataset
    val_dataset   : JailbreakDataset
    seed          : int
    arm_name      : str  e.g. "arm_a"

    Returns
    -------
    Trainer with best checkpoint loaded
    """
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    output_dir = str(MODEL_DIR / f"distilbert_{arm_name}_seed{seed}")
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        seed=seed,
        logging_steps=50,
        report_to="none",
    )
    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )
    print(f"Training: {arm_name.upper()}  |  Seed: {seed}")
    trainer.train()
    return trainer

### Section 5 - Multi-Seed Training Loop

Trains both arms across 3 seeds = **6 total training runs**.

Expected runtime: ~15-30 min per run on Apple Silicon MPS (~1.5-3 hours total).

In [ ]:
trainers_a = {}
trainers_b = {}

for seed in SEEDS:
    trainers_a[seed] = train_arm(train_dataset_a, val_dataset_a, seed, "arm_a")

print("Arm A (single-turn) training complete.")

In [ ]:
for seed in SEEDS:
    trainers_b[seed] = train_arm(train_dataset_b, val_dataset_b, seed, "arm_b")

print("Arm B (multi-turn) training complete.")

### Section 6 - Evaluation on T1 (Primary Multi-Turn Test Set)

All 6 models evaluated on the **shared multi-turn test set T1**.

Primary metric: **Unsafe Recall at fixed Precision = 0.95**

In [ ]:
def recall_at_precision(y_true, y_score, target_precision: float = 0.95) -> float:
    """Highest recall achievable at precision >= target_precision on the PR curve."""
    precisions, recalls, _ = precision_recall_curve(y_true, y_score)
    valid_recalls = recalls[precisions >= target_precision]
    return float(valid_recalls.max()) if len(valid_recalls) > 0 else 0.0


def bootstrap_ci(y_true, y_score, metric_fn, n_boot: int = 1000, ci: float = 0.95):
    """Bootstrap confidence interval for a scalar metric."""
    rng = np.random.default_rng(0)
    n = len(y_true)
    scores = [
        metric_fn(y_true[idx], y_score[idx])
        for idx in (rng.integers(0, n, n) for _ in range(n_boot))
    ]
    alpha = (1 - ci) / 2
    return np.percentile(scores, [alpha * 100, (1 - alpha) * 100])


def evaluate_on_t1(trainer: Trainer, y_true: np.ndarray) -> dict:
    """Predict on T1 and compute full metric suite."""
    preds_output = trainer.predict(test_dataset_t1)
    logits = preds_output.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    y_score = probs[:, 1]
    y_pred = (y_score >= 0.5).astype(int)

    precisions, recalls, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recalls, precisions)
    recall_95 = recall_at_precision(y_true, y_score)

    f1_ci  = bootstrap_ci(y_true, y_score,
                          lambda yt, ys: f1_score(yt, (ys >= 0.5).astype(int), average="macro"))
    roc_ci = bootstrap_ci(y_true, y_score, roc_auc_score)

    return {
        "recall_at_p95": recall_95,
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_unsafe": f1_score(y_true, y_pred, pos_label=1, average="binary"),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": pr_auc,
        "f1_macro_ci": f1_ci.tolist(),
        "roc_auc_ci": roc_ci.tolist(),
        "y_score": y_score,
        "y_pred": y_pred,
    }

In [ ]:
y_true_t1 = test_t1["harm"].astype(int).values

results_a = {seed: evaluate_on_t1(trainer, y_true_t1) for seed, trainer in trainers_a.items()}
results_b = {seed: evaluate_on_t1(trainer, y_true_t1) for seed, trainer in trainers_b.items()}

rows = []
for seed in SEEDS:
    for arm, results in [("A (single-turn)", results_a), ("B (multi-turn)", results_b)]:
        r = results[seed]
        rows.append({
            "Arm": arm, "Seed": seed,
            "Recall@P=0.95": round(r["recall_at_p95"], 4),
            "F1-Macro": round(r["f1_macro"], 4),
            "F1-Unsafe": round(r["f1_unsafe"], 4),
            "ROC-AUC": round(r["roc_auc"], 4),
            "PR-AUC": round(r["pr_auc"], 4),
        })

pd.DataFrame(rows).pipe(print)

In [ ]:
# Hypothesis Test
# H0: Model A and Model B perform identically on multi-turn jailbreak detection
# Decision rule: reject H0 if median(Recall_B) - median(Recall_A) > sigma(Recall_A)

recall_a_seeds = np.array([results_a[s]["recall_at_p95"] for s in SEEDS])
recall_b_seeds = np.array([results_b[s]["recall_at_p95"] for s in SEEDS])

median_a = np.median(recall_a_seeds)
median_b = np.median(recall_b_seeds)
sigma_a  = np.std(recall_a_seeds)
delta    = median_b - median_a
reject   = delta > sigma_a

print("Hypothesis Test: Recall @ P=0.95")
print(f"  Arm A recalls : {recall_a_seeds}")
print(f"  Arm B recalls : {recall_b_seeds}")
print(f"  Median A      : {median_a:.4f}")
print(f"  Median B      : {median_b:.4f}")
print(f"  sigma(A)      : {sigma_a:.4f}")
print(f"  Delta         : {delta:.4f}")
print(f"  Reject H0     : {reject}")
print()
if reject:
    print("REJECT H0 - Multi-turn training provides meaningful uplift.")
else:
    print("FAIL TO REJECT H0 - No significant uplift from multi-turn training.")

### Section 7 - Visualization & Export

In [ ]:
best_seed_a = max(SEEDS, key=lambda s: results_a[s]["recall_at_p95"])
best_seed_b = max(SEEDS, key=lambda s: results_b[s]["recall_at_p95"])

y_score_a = results_a[best_seed_a]["y_score"]
y_score_b = results_b[best_seed_b]["y_score"]

# Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, y_score) in zip(axes, [
        (f"Arm A - seed {best_seed_a}", y_score_a),
        (f"Arm B - seed {best_seed_b}", y_score_b)]):
    precs, recs, _ = precision_recall_curve(y_true_t1, y_score)
    ax.plot(recs, precs, lw=2, label=label)
    ax.axvline(x=recall_at_precision(y_true_t1, y_score), color="red",
               linestyle="--", alpha=0.7, label="Recall @ P=0.95")
    ax.axhline(y=0.95, color="gray", linestyle=":", alpha=0.7, label="P = 0.95")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"PR Curve - {label}")
    ax.legend(fontsize=8); ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

plt.suptitle("Precision-Recall Curves (T1 Multi-Turn Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pr_curve_distilbert.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pr_curve_distilbert.png")

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(7, 5))
for label, y_score in [(f"Arm A (seed {best_seed_a})", y_score_a),
                        (f"Arm B (seed {best_seed_b})", y_score_b)]:
    fpr, tpr, _ = roc_curve(y_true_t1, y_score)
    ax.plot(fpr, tpr, lw=2, label=f"{label}  (AUC={auc(fpr, tpr):.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random baseline")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves - DistilBERT (T1 Multi-Turn Test Set)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "roc_curve_distilbert.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: roc_curve_distilbert.png")

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, y_pred) in zip(axes, [
        (f"Arm A (seed {best_seed_a})", results_a[best_seed_a]["y_pred"]),
        (f"Arm B (seed {best_seed_b})", results_b[best_seed_b]["y_pred"])]):
    cm = confusion_matrix(y_true_t1, y_pred)
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(f"Confusion Matrix
{label}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Safe", "Unsafe"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Safe", "Unsafe"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix_distilbert.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrix_distilbert.png")

In [ ]:
# Export Metrics JSON
output_metrics = {
    "model": MODEL_NAME, "seeds": SEEDS,
    "hypothesis_test": {
        "median_recall_a": round(float(median_a), 4),
        "median_recall_b": round(float(median_b), 4),
        "sigma_recall_a": round(float(sigma_a), 4),
        "delta": round(float(delta), 4),
        "reject_h0": bool(reject),
    },
    "arm_a": {
        str(seed): {k: v.tolist() if isinstance(v, __import__("numpy").ndarray) else v
                    for k, v in results_a[seed].items() if k not in ("y_score", "y_pred")}
        for seed in SEEDS
    },
    "arm_b": {
        str(seed): {k: v.tolist() if isinstance(v, __import__("numpy").ndarray) else v
                    for k, v in results_b[seed].items() if k not in ("y_score", "y_pred")}
        for seed in SEEDS
    },
}
out_path = REPORTS_DIR / "distilbert_results.json"
with open(out_path, "w") as f:
    json.dump(output_metrics, f, indent=2)
print(f"Results saved to: {out_path}")
print(json.dumps(output_metrics["hypothesis_test"], indent=2))